## Get mobilenet_v3_small.tflite

In [ ]:
# ============================================================
# CELL A
# MOUNT DRIVE + INSTALL EXACT PACKAGE VERSIONS
# (pinned to avoid onnx2tf / tensorflow version conflicts)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch torchvision --upgrade
!pip install -q onnx==1.16.1
!pip install -q onnxruntime==1.18.1
!pip install -q onnxscript
!pip install -q onnxsim
!pip install -q sng4onnx
!pip install -q onnx_graphsurgeon
!pip install -q ai-edge-litert
!pip install -q onnx2tf

import os

SAVE_DIR = "/content/drive/MyDrive/TinyBayes/run_2000"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Setup complete. SAVE_DIR:", SAVE_DIR)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# CELL B
# LOAD PRETRAINED (NOT FINE-TUNED) MOBILENETV3-SMALL
# STRIP CLASSIFIER -> 576-D FEATURE EXTRACTOR
# ============================================================

import torch
import torch.nn as nn

from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights
)

device = torch.device("cpu")  # export in CPU mode for determinism

model = mobilenet_v3_small(
    weights=MobileNet_V3_Small_Weights.DEFAULT
)

model.classifier = nn.Identity()  # remove ImageNet head -> 576-d output
model.eval()
model.to(device)

# ------------------------------------------------------------
# Sanity check: confirm feature dimension is really 576
# ------------------------------------------------------------

dummy = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    test_feat = model(dummy)

print("Output shape:", test_feat.shape)

assert test_feat.shape[1] == 576, (
    f"Expected 576-d output, got {test_feat.shape[1]}. "
    "Do not proceed until this matches."
)

print("Verified: pretrained MobileNetV3-Small produces 576-d features.")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 74.4MB/s]


Output shape: torch.Size([1, 576])
Verified: pretrained MobileNetV3-Small produces 576-d features.


In [ ]:
# ============================================================
# CELL C
# EXPORT TO ONNX + VERIFY NUMERICALLY AGAINST PYTORCH
# ============================================================

import os
import numpy as np
import onnx
import onnxruntime as ort

ONNX_PATH = os.path.join(SAVE_DIR, "mobilenet_v3_small.onnx")

dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    input_names=["input"],
    output_names=["features"],
    opset_version=18,
    dynamo=False,  # use the stable TorchScript-based exporter
)

print("Saved ONNX:", ONNX_PATH)
print("File size (MB):", round(os.path.getsize(ONNX_PATH) / 1024 / 1024, 3))

# ------------------------------------------------------------
# Structural check
# ------------------------------------------------------------

onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("ONNX structural check: PASSED")
print("Opset used:", onnx_model.opset_import[0].version)

# ------------------------------------------------------------
# Numerical check: PyTorch vs ONNX Runtime, same random input
# ------------------------------------------------------------

x = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    pt_out = model(x).numpy()

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
onnx_out = session.run(None, {"input": x.numpy()})[0]

max_diff = np.max(np.abs(pt_out - onnx_out))
print("PyTorch shape:", pt_out.shape, "| ONNX shape:", onnx_out.shape)
print("Max abs difference:", max_diff)

assert pt_out.shape == onnx_out.shape, "Shape mismatch between PyTorch and ONNX!"
assert max_diff < 1e-4, (
    f"ONNX output diverges from PyTorch by {max_diff}. "
    "Do not proceed to TFLite conversion until this passes."
)

print("Verified: ONNX export matches PyTorch numerically.")

/tmp/ipykernel_1483/108139554.py:15: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Saved ONNX: /content/drive/MyDrive/TinyBayes/run_2000/mobilenet_v3_small.onnx
File size (MB): 3.545
ONNX structural check: PASSED
Opset used: 18
PyTorch shape: (1, 576) | ONNX shape: (1, 576)
Max abs difference: 1.66893e-06
Verified: ONNX export matches PyTorch numerically.


In [ ]:
# ============================================================
# CELL D
# CONVERT ONNX -> TENSORFLOW SAVEDMODEL
# ============================================================

from onnx2tf import convert

SAVED_MODEL_DIR = os.path.join(SAVE_DIR, "mobilenet_saved_model")

convert(
    input_onnx_file_path=ONNX_PATH,
    output_folder_path=SAVED_MODEL_DIR,
    copy_onnx_input_output_names_to_tflite=True,
)

# ------------------------------------------------------------
# Verify the SavedModel folder actually has content
# ------------------------------------------------------------

assert os.path.exists(SAVED_MODEL_DIR), "SavedModel folder was not created."

found_files = []
for root, dirs, files in os.walk(SAVED_MODEL_DIR):
    for f in files:
        found_files.append(os.path.join(root, f))

print(f"SavedModel folder contains {len(found_files)} files:")
for f in found_files:
    print(" ", f)

assert len(found_files) > 0, "SavedModel folder is empty — conversion failed."
print("\nVerified: SavedModel created successfully.")


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                   ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add               │ 6              │ 6                │
│ Constant          │ 104            │ 104              │
│ Conv              │ 52             │ 52               │
│ Flatten           │ 1              │ 1                │
│ GlobalAveragePool │ 10             │ 10               │
│ HardSigmoid       │ 9              │ 9                │
│ HardSwish         │ 18             │ 18               │
│ Mul               │ 9              │ 9                │
│ Relu              │ 14             │ 14               │
│ Model Size        │ 3.5MiB         │ 3.6MiB           │
└───────────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━

flatbuffer_direct lowering:   0%|          | 0/119 [00:00<?, ?it/s]

flatbuffer_direct post-lowering:   0%|          | 0/6 [00:00<?, ?it/s]

flatbuffer_direct export:   0%|          | 0/3 [00:00<?, ?it/s]

flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.086s serialize=0.028s (sanitize=0.000s build=0.003s pack=0.024s output=0.001s) write=0.054s size=3.56MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct total=0.048s serialize=0.023s (sanitize=0.000s build=0.003s pack=0.020s output=0.000s) write=0.024s size=1.81MB
Float32 tflite output complete! (/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_saved_model/mobilenet_v3_small_float32.tflite)
Float16 tflite output complete! (/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_saved_model/mobilenet_v3_small_float16.tflite)
Tensor correspondence report output complete! (/content/drive/MyDrive/TinyBayes/run_2000/mobilenet_saved_model/mobilenet_v3_small_tensor_correspondence_report.json)
Input/Output tensor names are directly written from ONNX graph in flatbuffer_direct backend.
SavedModel folder contains 5 files:
  /content/drive/MyDrive/TinyBayes/run_2000/mobilenet_saved_model/schema_generated.py


In [ ]:
# ============================================================
# CELL E (REVISED)
# onnx2tf already produced TFLite directly (flatbuffer_direct path)
# Just locate and copy it to the expected filename
# ============================================================

import shutil

# onnx2tf's direct-export path already created these:
FLOAT32_TFLITE = os.path.join(SAVED_MODEL_DIR, "mobilenet_v3_small_float32.tflite")
FLOAT16_TFLITE = os.path.join(SAVED_MODEL_DIR, "mobilenet_v3_small_float16.tflite")

assert os.path.exists(FLOAT32_TFLITE), f"Expected file not found: {FLOAT32_TFLITE}"

TFLITE_PATH = os.path.join(SAVE_DIR, "mobilenet_v3_small.tflite")
shutil.copy(FLOAT32_TFLITE, TFLITE_PATH)

print("Copied float32 TFLite to:", TFLITE_PATH)
print("File size (MB):", round(os.path.getsize(TFLITE_PATH) / 1024 / 1024, 3))

# ------------------------------------------------------------
# Verify it loads and has the expected shape
# ------------------------------------------------------------

import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\nInput details:", input_details[0]["shape"], input_details[0]["dtype"])
print("Output details:", output_details[0]["shape"], output_details[0]["dtype"])

assert output_details[0]["shape"][-1] == 576, (
    f"Expected TFLite output dim 576, got {output_details[0]['shape'][-1]}."
)

print("\nVerified: TFLite model loads correctly with 576-d output.")

Copied float32 TFLite to: /content/drive/MyDrive/TinyBayes/run_2000/mobilenet_v3_small.tflite
File size (MB): 3.564

Input details: [  1 224 224   3] <class 'numpy.float32'>
Output details: [  1 576] <class 'numpy.float32'>

Verified: TFLite model loads correctly with 576-d output.


In [ ]:
# ============================================================
# CELL F
# FINAL VERIFICATION: TFLITE OUTPUT vs PYTORCH OUTPUT
# ON A REAL IMAGE (NOT JUST RANDOM NOISE)
# ============================================================

import cv2
from torchvision import transforms

# Grab any one real image from the training folder to sanity-check on
IMAGE_DIR_TRAIN = os.path.join(
    DATASET_ROOT if "DATASET_ROOT" in dir() else "/content/drive/MyDrive/TinyBayes/amini_dataset",
    "dataset", "images", "train"
)

sample_image_name = os.listdir(IMAGE_DIR_TRAIN)[0]
sample_image_path = os.path.join(IMAGE_DIR_TRAIN, sample_image_name)

image = cv2.imread(sample_image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# --- PyTorch path ---
transform_mobile = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

img_tensor = transform_mobile(image).unsqueeze(0)

with torch.no_grad():
    pt_feat = model(img_tensor).numpy().flatten()

# --- TFLite path (matches your Step 4 preprocessing exactly) ---
img_tf = cv2.resize(image, (224, 224)).astype(np.float32)
img_tf /= 255.0
img_tf[:, :, 0] = (img_tf[:, :, 0] - 0.485) / 0.229
img_tf[:, :, 1] = (img_tf[:, :, 1] - 0.456) / 0.224
img_tf[:, :, 2] = (img_tf[:, :, 2] - 0.406) / 0.225
input_tensor = np.expand_dims(img_tf, axis=0).astype(np.float32)

interpreter.set_tensor(input_details[0]["index"], input_tensor)
interpreter.invoke()
tflite_feat = interpreter.get_tensor(output_details[0]["index"])[0]

max_diff = np.max(np.abs(pt_feat - tflite_feat))
cosine = np.dot(pt_feat, tflite_feat) / (
    np.linalg.norm(pt_feat) * np.linalg.norm(tflite_feat)
)

print("Tested on real image:", sample_image_name)
print("Max abs difference:", max_diff)
print("Cosine similarity:", cosine)

if max_diff < 1e-2 and cosine > 0.999:
    print("\n✅ TFLite model verified against PyTorch on a real image.")
else:
    print("\n⚠️ WARNING: TFLite output diverges more than expected.")
    print("Do not trust downstream Jacobi-DMR results until this is resolved.")

Tested on real image: ID_uz8ZPb.jpg
Max abs difference: 1.2395506
Cosine similarity: 0.9493993

⚠️ WARNING: TFLite output diverges more than expected.
Do not trust downstream Jacobi-DMR results until this is resolved.
